In [17]:
from dataclasses import asdict

from pyproj import Transformer
import cv2
import numpy as np
import matplotlib.pyplot as plt
from lightglue import viz2d

from uavcalibration.calibration import Calibration
from uavcalibration.datasets import VisLocDataset
from uavcalibration.transform import *

dataset = VisLocDataset(r"..\datasets\UAV_VisLoc_example")

In [18]:
uav_data = dataset[50]
calibration = Calibration(uav_data.uav_image)
calibration.coarse_calibrate(**asdict(uav_data))
src_shape = uav_data.uav_image.shape
calibration.transform.adjust_shape(src_shape=(src_shape[1],src_shape[0]))

In [19]:
h, w = calibration.uav_image.shape[:2]
corner = np.array([[0, 0], [w, 0], [w, h], [0, h]], np.float64)
corner_utm = cv2.perspectiveTransform(
    corner.reshape(1, -1, 2), calibration.transform.combined.mat
).reshape(-1, 2)
transformer = Transformer.from_crs(
    calibration.transform.crs.crs, "epsg:4326", always_xy=True
)
corner_lonlat = np.stack(transformer.transform(corner_utm[:, 0], corner_utm[:, 1]), 1)
corner_lonlat = np.stack([corner_lonlat.min(0), corner_lonlat.max(0)])
print(corner_lonlat.ravel())
# fetch satellite image here

[119.85438835  32.32889946 119.8586516   32.33361637]


In [20]:
calibration.fine_calibrate(
    satellite_image=uav_data.satellite_image,
    satellite_crs=CRSTransform(uav_data.satellite_transform),
)
# viz2d.plot_images([calibration.calibrated_image, uav_data.satellite_image])

In [ ]:
# 全局变量存储鼠标位置
uav_pos = (0, 0)
satellite_pos = (0, 0)
satellite_lonlat = (0, 0)
data_index = 0


def calibrate():
    calibration = Calibration(uav_data.uav_image)
    calibration.coarse_calibrate(**asdict(uav_data))
    src_shape = uav_data.uav_image.shape
    calibration.transform.adjust_shape(src_shape=(src_shape[1], src_shape[0]))
    calibration.fine_calibrate(
        satellite_image=uav_data.satellite_image,
        satellite_crs=CRSTransform(uav_data.satellite_transform),
    )
    return calibration


# 鼠标回调函数
def mouse_callback(event, x, y, flags, param):
    global uav_pos, satellite_pos, satellite_lonlat
    if event == cv2.EVENT_MOUSEMOVE:  # 鼠标移动事件
        uav_pos = (x, y)
        satellite_pos = tuple(
            calibration.transform.apply(np.array(uav_pos, np.float64))
            .ravel()
            .astype(int)
        )
        satellite_lonlat = tuple(
            calibration.transform.crs.apply(np.array(satellite_pos, np.float64)).ravel()
        )


size = 10

# 创建窗口并绑定回调函数
cv2.namedWindow("UAV Image", cv2.WINDOW_KEEPRATIO)
cv2.namedWindow("Satellite Image", cv2.WINDOW_KEEPRATIO)
cv2.setMouseCallback("UAV Image", mouse_callback)

uav_data = dataset[data_index]
calibration = calibrate()
while True:
    # 在图像上绘制坐标（可选）
    uav_image = uav_data.uav_image[..., ::-1].copy()
    satellite_image = uav_data.satellite_image[..., ::-1].copy()
    cv2.putText(
        uav_image,
        f"Position: {uav_pos}",
        (size * 5, size * 15),
        cv2.FONT_HERSHEY_SIMPLEX,
        size * 0.3,
        (0, 255, 0),
        size,
    )
    cv2.putText(
        satellite_image,
        f"lonlat: {uav_pos}",
        (size * 5, size * 15),
        cv2.FONT_HERSHEY_SIMPLEX,
        size * 0.3,
        (0, 255, 0),
        size,
    )
    cv2.circle(satellite_image, satellite_pos, size, (0, 255, 0), size)
    # 显示图像
    cv2.imshow("UAV Image", uav_image)
    cv2.imshow("Satellite Image", satellite_image)
    key = cv2.waitKey(1)
    # 按ESC键退出
    if key == 27:
        break
    elif key in [119, 97, 115, 100]:
        if key in [
            119,
        ]:  # w
            data_index -= 10
        elif key in [97]:  # a
            data_index -= 1
        elif key in [115]:  # s
            data_index += 10
        elif key in [100]:  # d
            data_index += 1
        uav_data = dataset[data_index]
        calibration = calibrate()
cv2.destroyAllWindows()